# Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

# Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [6]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
model= init_chat_model("llama-3.3-70b-versatile",
                       model_provider= "groq"
                       )
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000184367B0770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018436C24640>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description=("The movie rating out of 10")
                       )

In [9]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000184367B0770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018436C24640>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': 

In [11]:
model.invoke("give me the details of movie Annabelle")

AIMessage(content="**Annabelle (2014)**\n\n**Overview:**\nAnnabelle is a supernatural horror film directed by John R. Leonetti, written by Gary Dauberman, and produced by Peter Safran and James Wan. The movie is a spin-off of The Conjuring (2013) and serves as the first installment in the Annabelle film series.\n\n**Plot:**\nThe film takes place in 1970 and follows the story of John Form (Ward Horton), a medical student, and his pregnant wife Mia (Annabelle Wallis). The couple is expecting their first child and is excited to start their new life together. One day, John surprises Mia with a vintage porcelain doll, which she had been searching for. However, their joy is short-lived, as they soon discover that the doll, Annabelle, is possessed by a malevolent spirit.\n\nAs strange and terrifying events begin to occur, the couple realizes that Annabelle is not just a simple doll but a conduit for a dark entity that seeks to harm them and their unborn child. The entity is revealed to be the

In [14]:
model_with_structure.invoke("give me the details of movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

# TypedDict

TypedDict provides a simpler alternative using python'built-in typing , ideal when you dont need runtimes validation.

In [15]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypeddict = model.with_structured_output(MovieDict)

response = model_withtypeddict.invoke(
    "Please provide the details of the movie avengers"
)

response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}

In [17]:
class Actor(TypedDict):
    name: str
    role: str


class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )


model_with_structure = model.with_structured_output(MovieDetails)


response = model_with_structure.invoke(
    "Provide details about the movie exorcism"
)

response

{'budget': 12000000,
 'cast': [{'name': 'Ellen Burstyn', 'role': 'Chris MacNeil'},
  {'name': 'Max von Sydow', 'role': 'Father Merrin'}],
 'genres': ['Horror'],
 'title': 'The Exorcism',
 'year': 1973}

In [18]:
 model.profile

{'name': 'Llama 3.3 70B Versatile',
 'release_date': '2024-12-06',
 'last_updated': '2024-12-06',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

# DataClasses

A data class is a class typically containing mainly data, although there aren't really any restriction. You create it using @dataclass decorator.

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


agent = create_agent(
    model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})


result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [28]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""

    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


agent = create_agent(
    model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')